In [ ]:
# ============================================================
# 0) GOOGLE DRIVE BAĞLANTISI
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
"""
ALL-IN-ONE Oxford Battery Degradation Dataset 1 SoH Experiments
================================================================
This script is designed for Google Colab.

What it does from scratch:
1) Downloads Oxford Battery Degradation Dataset 1 .mat file if not available.
2) Extracts Multi-HI features under partial charging voltage windows.
3) Cleans feature leakage in model inputs.
4) Runs leave-one-cell-out experiments.
5) Compares Single-HI EKF, approximate paper-style GPR-EKF, ML models, and deep models.
6) Runs repeated Missing-HI robustness tests.
7) Generates SHAP bar, beeswarm, and dependence plots.
8) Saves CSV/XLSX results under /content/oxford_advanced_results.

Note:
- Deep learning models can take time. Set RUN_DEEP_MODELS = False in the advanced section if needed.
- The script uses SoH/capacity only as target/ground-truth, not as ML input features.
"""

# ============================================================
# PART A - DOWNLOAD DATASET AND EXTRACT MULTI-HI FEATURES
# ============================================================

import os
import re
import sys
import math
import warnings
import subprocess
import importlib.util
from pathlib import Path

warnings.filterwarnings("ignore")


def pip_install_if_missing(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg, imp in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("scikit-learn", "sklearn"),
    ("tqdm", "tqdm"),
    ("requests", "requests"),
]:
    pip_install_if_missing(pkg, imp)

import numpy as np
import pandas as pd
import requests
import scipy.io as sio
from scipy.io.matlab import mat_struct
from scipy.signal import savgol_filter
from tqdm.auto import tqdm

try:
    from IPython.display import display
except Exception:
    def display(obj):
        try:
            print(obj.to_string() if hasattr(obj, "to_string") else obj)
        except Exception:
            print(obj)

In [ ]:
# ============================================================
# DRIVE-BASED PROJECT PATHS
# ============================================================
# Not: /content geçici Colab alanıdır. Kalıcı kayıt için tüm ana çıktıları
# /content/drive/MyDrive altına yönlendiriyoruz.
DRIVE_ROOT = "/content/drive/MyDrive/Batarya"
PROJECT_DIR = os.path.join(DRIVE_ROOT, "Oxford_Battery_Degradation_Dataset_2")
BASE_DIR = os.path.join(PROJECT_DIR, "data")
FEATURE_DIR = os.path.join(PROJECT_DIR, "features")

for _dir in [PROJECT_DIR, BASE_DIR, FEATURE_DIR]:
    os.makedirs(_dir, exist_ok=True)

MAT_PATH = os.path.join(BASE_DIR, "Oxford_Battery_Degradation_Dataset_1.mat")
README_PATH = os.path.join(BASE_DIR, "Readme.txt")
FEATURE_OUT_CSV = os.path.join(FEATURE_DIR, "multi_hi_features.csv")

print("Project directory:", PROJECT_DIR)
print("Data directory:", BASE_DIR)
print("Feature output CSV:", FEATURE_OUT_CSV)

DATA_URL = "https://ora.ox.ac.uk/objects/uuid%3A03ba4b01-cfed-46d3-9b1a-7d4a7bdf6fac/files/m5ac36a1e2073852e4f1f7dee647909a7"
README_URL = "https://ora.ox.ac.uk/objects/uuid%3A03ba4b01-cfed-46d3-9b1a-7d4a7bdf6fac/files/m43cc05e7c5f1245f4895d9dbd495e52f"

PARTIAL_WINDOWS = {
    "full_36_42": (3.60, 4.20),
    "medium_37_40": (3.70, 4.00),
    "narrow_375_395": (3.75, 3.95),
    "very_narrow_38_39": (3.80, 3.90),
}


def download_file(url, out_path, chunk_size=1024 * 1024):
    if os.path.exists(out_path) and os.path.getsize(out_path) > 1024:
        print(f"Already exists: {out_path}")
        return
    print(f"Downloading: {out_path}")
    r = requests.get(url, stream=True, timeout=60)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(out_path, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=os.path.basename(out_path)) as pbar:
        for chunk in r.iter_content(chunk_size=chunk_size):
            if chunk:
                f.write(chunk)
                pbar.update(len(chunk))


def find_or_download_dataset():
    candidates = [
        MAT_PATH,  # preferred: Drive project data folder
        "/content/drive/MyDrive/Oxford_Battery_Degradation_Dataset_1.mat",
        "/content/Oxford_Battery_Degradation_Dataset_1.mat",
        "Oxford_Battery_Degradation_Dataset_1.mat",
    ]
    for p in candidates:
        if os.path.exists(p) and os.path.getsize(p) > 1024:
            if p != MAT_PATH:
                print(f"Dataset found at {p}. Using this path.")
                return p
            return MAT_PATH
    try:
        download_file(README_URL, README_PATH)
        download_file(DATA_URL, MAT_PATH)
        return MAT_PATH
    except Exception as e:
        print("Automatic download failed.")
        print("Error:", repr(e))
        print("Manual solution: download Oxford_Battery_Degradation_Dataset_1.mat from Oxford ORA and upload it to Colab.")
        print("Expected path:", MAT_PATH)
        raise


def matlab_to_python(obj):
    if isinstance(obj, mat_struct):
        return {field_name: matlab_to_python(getattr(obj, field_name)) for field_name in obj._fieldnames}
    if isinstance(obj, np.ndarray):
        if obj.dtype == object:
            if obj.size == 1:
                return matlab_to_python(obj.item())
            return np.array([matlab_to_python(x) for x in obj.flat], dtype=object).reshape(obj.shape)
        if obj.dtype.names is not None:
            if obj.size == 1:
                item = obj.item()
                return {name: matlab_to_python(item[name]) for name in obj.dtype.names}
            return [matlab_to_python(x) for x in obj]
        return np.array(obj).squeeze()
    return obj


def cycle_to_number(cycle_name):
    m = re.search(r"cyc(\d+)", str(cycle_name))
    return int(m.group(1)) if m else np.nan


def to_1d_float(arr):
    arr = np.array(arr).squeeze().astype(float)
    return arr[np.isfinite(arr)]


def get_channel(cycle_dict, mode, key):
    try:
        return to_1d_float(cycle_dict[mode][key])
    except Exception:
        return np.array([])


def safe_get_channel(cycle_dict, mode, key_candidates):
    if isinstance(key_candidates, str):
        key_candidates = [key_candidates]
    for key in key_candidates:
        arr = get_channel(cycle_dict, mode, key)
        if arr is not None and len(arr) > 0:
            return arr
    return np.array([])


def compute_capacity_from_discharge(cycle_dict):
    q = safe_get_channel(cycle_dict, "C1dc", ["q", "Q", "capacity", "Capacity"])
    if len(q) < 10:
        return np.nan
    return float(abs(np.nanmax(q) - np.nanmin(q)))


def get_charge_arrays(cycle_dict):
    v = safe_get_channel(cycle_dict, "C1ch", ["v", "V", "voltage", "Voltage"])
    q = safe_get_channel(cycle_dict, "C1ch", ["q", "Q", "capacity", "Capacity"])
    t = safe_get_channel(cycle_dict, "C1ch", ["t", "Tt", "time", "Time"])
    T = safe_get_channel(cycle_dict, "C1ch", ["T", "temp", "Temp", "temperature", "Temperature"])
    if len(v) == 0 or len(q) == 0:
        return np.array([]), np.array([]), np.array([]), np.array([])
    n = min(len(v), len(q))
    v = np.asarray(v[:n], dtype=float)
    q = np.asarray(q[:n], dtype=float)
    t = np.asarray(t[:n], dtype=float) if len(t) >= n else np.full(n, np.nan)
    T = np.asarray(T[:n], dtype=float) if len(T) >= n else np.full(n, np.nan)
    mask = np.isfinite(v) & np.isfinite(q)
    return v[mask], q[mask], t[mask], T[mask]


def compute_ica_and_dva_window(cycle_dict, v_window, n_grid=500, smooth_window=31, smooth_poly=3):
    v, q, t, T = get_charge_arrays(cycle_dict)
    if len(v) < 30:
        return None, None, None, None, None
    lo, hi = v_window
    mask = np.isfinite(v) & np.isfinite(q) & (v >= lo) & (v <= hi)
    if mask.sum() < 30:
        return None, None, None, None, None
    v_w, q_w = v[mask], q[mask]
    t_w = t[mask] if len(t) == len(v) else np.full(mask.sum(), np.nan)
    T_w = T[mask] if len(T) == len(v) else np.full(mask.sum(), np.nan)
    df_tmp = pd.DataFrame({"v": v_w, "q": q_w, "t": t_w, "T": T_w}).replace([np.inf, -np.inf], np.nan).dropna(subset=["v", "q"])
    if len(df_tmp) < 30:
        return None, None, None, None, None
    df_tmp["v_round"] = df_tmp["v"].round(4)
    df_g = df_tmp.groupby("v_round", as_index=False).agg(q=("q", "mean"), t=("t", "mean"), T=("T", "mean"))
    df_g = df_g.rename(columns={"v_round": "v"}).sort_values("v")
    v_unique = df_g["v"].values.astype(float)
    q_unique = df_g["q"].values.astype(float)
    if len(v_unique) < 20:
        return None, None, None, None, None
    v_grid = np.linspace(v_unique.min(), v_unique.max(), n_grid)
    q_grid = np.interp(v_grid, v_unique, q_unique)
    sw = smooth_window
    if sw >= len(q_grid):
        sw = len(q_grid) - 1
    if sw % 2 == 0:
        sw -= 1
    if sw < 7:
        sw = 7
    try:
        q_smooth = savgol_filter(q_grid, sw, smooth_poly)
    except Exception:
        q_smooth = q_grid
    ic = np.gradient(q_smooth, v_grid)
    dq = np.gradient(q_smooth)
    dv = np.gradient(v_grid)
    with np.errstate(divide="ignore", invalid="ignore"):
        dva = dv / dq
    try:
        ic = savgol_filter(ic, sw, smooth_poly)
    except Exception:
        pass
    ic[~np.isfinite(ic)] = np.nan
    dva[~np.isfinite(dva)] = np.nan
    return v_grid, q_grid, ic, dva, df_tmp


def peak_width_half_max(v_grid, ic):
    if v_grid is None or ic is None:
        return np.nan
    mask = np.isfinite(ic) & (ic > 0)
    if mask.sum() < 5:
        return np.nan
    v = v_grid[mask]
    y = ic[mask]
    y_max = np.nanmax(y)
    if not np.isfinite(y_max) or y_max <= 0:
        return np.nan
    above = y >= 0.5 * y_max
    if above.sum() < 2:
        return np.nan
    return float(v[above].max() - v[above].min())


def extract_multi_hi_for_window(cycle_dict, v_window):
    v_grid, q_grid, ic, dva, raw_df = compute_ica_and_dva_window(cycle_dict, v_window)
    out = {
        "pmax": np.nan,
        "pmax_voltage": np.nan,
        "ic_area": np.nan,
        "ic_mean": np.nan,
        "ic_std": np.nan,
        "peak_width": np.nan,
        "q_delta_window": np.nan,
        "charge_time_window": np.nan,
        "dva_mean": np.nan,
        "dva_std": np.nan,
        "dva_abs_mean": np.nan,
        "temp_mean_window": np.nan,
        "temp_max_window": np.nan,
    }
    if v_grid is None:
        return out
    mask_ic = np.isfinite(ic) & (ic > 0)
    if mask_ic.sum() >= 5:
        ic_pos = ic[mask_ic]
        v_pos = v_grid[mask_ic]
        upper = np.nanpercentile(ic_pos, 99.5)
        ic_clip = np.clip(ic_pos, None, upper)
        idx = int(np.nanargmax(ic_clip))
        out["pmax"] = float(ic_clip[idx])
        out["pmax_voltage"] = float(v_pos[idx])
        out["ic_area"] = float(np.trapz(ic_clip, v_pos))
        out["ic_mean"] = float(np.nanmean(ic_clip))
        out["ic_std"] = float(np.nanstd(ic_clip))
        out["peak_width"] = peak_width_half_max(v_grid, ic)
    if q_grid is not None and np.isfinite(q_grid).sum() > 5:
        out["q_delta_window"] = float(np.nanmax(q_grid) - np.nanmin(q_grid))
    if raw_df is not None and len(raw_df) > 0:
        if "t" in raw_df.columns and raw_df["t"].notna().sum() > 2:
            out["charge_time_window"] = float(raw_df["t"].max() - raw_df["t"].min())
        if "T" in raw_df.columns and raw_df["T"].notna().sum() > 2:
            out["temp_mean_window"] = float(raw_df["T"].mean())
            out["temp_max_window"] = float(raw_df["T"].max())
    if dva is not None:
        mask_dva = np.isfinite(dva)
        if mask_dva.sum() > 5:
            dva_sel = dva[mask_dva]
            lo, hi = np.nanpercentile(dva_sel, [1, 99])
            dva_clip = np.clip(dva_sel, lo, hi)
            out["dva_mean"] = float(np.nanmean(dva_clip))
            out["dva_std"] = float(np.nanstd(dva_clip))
            out["dva_abs_mean"] = float(np.nanmean(np.abs(dva_clip)))
    return out


def detect_soh_outliers(df, threshold=0.04):
    df = df.sort_values(["cell", "cycle_number"]).copy()
    df["soh_rolling_median"] = np.nan
    df["soh_outlier"] = False
    df["soh_delta"] = np.nan
    for cell_name, g in df.groupby("cell"):
        idx = g.index
        s = g["soh"].astype(float)
        med = s.rolling(window=5, center=True, min_periods=2).median()
        delta = s.diff()
        outlier = (np.abs(s - med) > threshold) | (delta < -threshold)
        df.loc[idx, "soh_rolling_median"] = med.values
        df.loc[idx, "soh_delta"] = delta.values
        df.loc[idx, "soh_outlier"] = outlier.fillna(False).values
    return df


def load_oxford_mat_as_dict(mat_path):
    print("Loading MAT file:", mat_path)
    mat = sio.loadmat(mat_path, squeeze_me=True, struct_as_record=False)
    data = {}
    for cell_idx in range(1, 9):
        cell_name = f"Cell{cell_idx}"
        if cell_name in mat:
            data[cell_name] = matlab_to_python(mat[cell_name])
            print(cell_name, "loaded. cycles:", len(data[cell_name].keys()))
        else:
            print(cell_name, "not found.")
    return data


def build_multi_hi_features(data):
    multi_rows = []
    for cell_name in sorted(data.keys()):
        cycles = sorted(data[cell_name].keys(), key=cycle_to_number)
        for cyc_name in tqdm(cycles, desc=f"Multi-HI {cell_name}"):
            cyc = data[cell_name][cyc_name]
            cycle_number = cycle_to_number(cyc_name)
            capacity_mAh = compute_capacity_from_discharge(cyc)
            T_ch = safe_get_channel(cyc, "C1ch", ["T", "temp", "Temp", "temperature", "Temperature"])
            temp_mean = np.nanmean(T_ch) if len(T_ch) > 0 else np.nan
            temp_max = np.nanmax(T_ch) if len(T_ch) > 0 else np.nan
            row = {
                "cell": cell_name,
                "cycle_name": cyc_name,
                "cycle_number": cycle_number,
                "capacity_mAh": capacity_mAh,
                "temp_mean": temp_mean,
                "temp_max": temp_max,
            }
            for tag, win in PARTIAL_WINDOWS.items():
                feats = extract_multi_hi_for_window(cyc, win)
                for k, v in feats.items():
                    row[f"{k}_{tag}"] = v
            multi_rows.append(row)
    multi_features = pd.DataFrame(multi_rows).sort_values(["cell", "cycle_number"]).reset_index(drop=True)
    multi_features["soh"] = np.nan
    for cell_name, g in multi_features.groupby("cell"):
        idx = g.index
        first_capacity = g["capacity_mAh"].dropna().iloc[0]
        multi_features.loc[idx, "soh"] = multi_features.loc[idx, "capacity_mAh"] / first_capacity
        for tag in PARTIAL_WINDOWS.keys():
            pcol = f"pmax_{tag}"
            ncol = f"pmax_norm_{tag}"
            if pcol in multi_features.columns and g[pcol].dropna().shape[0] > 0:
                first_pmax = g[pcol].dropna().iloc[0]
                multi_features.loc[idx, ncol] = multi_features.loc[idx, pcol] / first_pmax
            else:
                multi_features.loc[idx, ncol] = np.nan
    multi_features = detect_soh_outliers(multi_features, threshold=0.04)
    return multi_features

In [ ]:
print("\n================ PART A: DATA + FEATURE EXTRACTION ================")
mat_path = find_or_download_dataset()
data = load_oxford_mat_as_dict(mat_path)
multi_features = build_multi_hi_features(data)
print("Multi-HI feature table shape:", multi_features.shape)
print(multi_features.head().to_string())

multi_features.to_csv(FEATURE_OUT_CSV, index=False)

# İsteğe bağlı Colab çalışma alanına da bir kopya bırakıyoruz; ana kalıcı dosya Drive'daki FEATURE_OUT_CSV'dir.
multi_features.to_csv("/content/multi_hi_features.csv", index=False)

print("Saved feature file to Google Drive:", FEATURE_OUT_CSV)
print("Temporary Colab copy:", "/content/multi_hi_features.csv")

# Quick correlation file for convenience
corr_rows = []
for col in multi_features.select_dtypes(include=[np.number]).columns:
    if col == "soh":
        continue
    valid = multi_features[["soh", col]].dropna()
    if len(valid) >= 10:
        corr = valid["soh"].corr(valid[col])
        corr_rows.append({"feature": col, "corr_with_soh": corr, "abs_corr": abs(corr), "valid_count": len(valid)})
corr_df = pd.DataFrame(corr_rows).sort_values("abs_corr", ascending=False)
corr_df.to_csv(os.path.join(FEATURE_DIR, "multi_hi_correlation_with_soh.csv"), index=False)
print("Saved correlation file to Google Drive:", os.path.join(FEATURE_DIR, "multi_hi_correlation_with_soh.csv"))
print("Top correlated features with SoH:")
print(corr_df.head(20).to_string(index=False))

In [ ]:
# ============================================================
# PART B - ADVANCED EXPERIMENTS
# ============================================================



# ============================================================
# ADVANCED OXFORD BATTERY SOH EXPERIMENTS
# Multi-HI + Partial Window + Leakage Cleaning + Single-HI EKF
# Approx. Paper-style GPR-EKF + ML models + LSTM/GRU/TCN/Transformer
# Missing-HI robustness + SHAP plots
# ============================================================

# ------------------------------------------------------------
# 0) COLAB INSTALLS
# ------------------------------------------------------------
import sys, subprocess, importlib.util, os, warnings, math, json, random

def pip_install_if_missing(pkg, import_name=None):
    import_name = import_name or pkg
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core packages
for pkg, imp in [
    ("numpy", "numpy"), ("pandas", "pandas"), ("scikit-learn", "sklearn"),
    ("matplotlib", "matplotlib"), ("seaborn", "seaborn"),
    ("xgboost", "xgboost"), ("lightgbm", "lightgbm"), ("catboost", "catboost"),
    ("shap", "shap"), ("openpyxl", "openpyxl")
]:
    try:
        pip_install_if_missing(pkg, imp)
    except Exception as e:
        print(f"WARNING: {pkg} could not be installed/imported: {e}")

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    def display(obj):
        try:
            print(obj.to_string() if hasattr(obj, "to_string") else obj)
        except Exception:
            print(obj)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel

try:
    from xgboost import XGBRegressor
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LGBM_AVAILABLE = True
except Exception:
    LGBM_AVAILABLE = False

try:
    from catboost import CatBoostRegressor
    CAT_AVAILABLE = True
except Exception:
    CAT_AVAILABLE = False

In [ ]:
# ------------------------------------------------------------
# 1) CONFIG
# ------------------------------------------------------------
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# If deep models are slow, set this to False.
RUN_DEEP_MODELS = True
DEEP_EPOCHS = 80
DEEP_SEQ_LEN = 5
DEEP_BATCH_SIZE = 32
DEEP_LR = 1e-3
DEEP_PATIENCE = 15

# Missing-HI robustness settings
MISSING_RANDOM_SEEDS = list(range(10))
MISSING_PROBS = [0.0, 0.2, 0.4, 0.6, 0.8]

# If True, outlier rows will be excluded in one of the repeated analyses.
EXCLUDE_OUTLIER_OPTIONS = [False, True]

# Partial-charging windows already encoded in multi_hi_features.csv
WINDOWS = {
    "full_36_42": "Full 3.60-4.20 V",
    "medium_37_40": "Medium 3.70-4.00 V",
    "narrow_375_395": "Narrow 3.75-3.95 V",
    "very_narrow_38_39": "Very narrow 3.80-3.90 V",
}

# Kalıcı sonuç klasörü: Google Drive
# Bu hücre Part A çalıştırılmadan tek başına çalıştırılırsa PROJECT_DIR'i tekrar tanımlar.
DRIVE_ROOT = globals().get("DRIVE_ROOT", "/content/drive/MyDrive")
PROJECT_DIR = globals().get("PROJECT_DIR", os.path.join(DRIVE_ROOT, "Oxford_Battery_Degradation_Dataset_2"))
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(RESULT_DIR, exist_ok=True)
print("Result directory:", RESULT_DIR)

In [ ]:
# ------------------------------------------------------------
# 2) LOAD MULTI-HI FEATURE FILE
# ------------------------------------------------------------
# This code continues from the previous notebook.
# It expects multi_hi_features.csv produced by your previous feature extraction.

DRIVE_ROOT = globals().get("DRIVE_ROOT", "/content/drive/MyDrive")
PROJECT_DIR = globals().get("PROJECT_DIR", os.path.join(DRIVE_ROOT, "Oxford_Battery_Degradation_Dataset_2"))
FEATURE_DIR = globals().get("FEATURE_DIR", os.path.join(PROJECT_DIR, "features"))
FEATURE_OUT_CSV = globals().get("FEATURE_OUT_CSV", os.path.join(FEATURE_DIR, "multi_hi_features.csv"))

CANDIDATE_FEATURE_PATHS = [
    FEATURE_OUT_CSV,  # preferred: Google Drive project feature file
    os.path.join(FEATURE_DIR, "multi_hi_features.csv"),
    "/content/multi_hi_features.csv",  # temporary Colab copy
    "multi_hi_features.csv",
    "/content/oxford_battery/multi_hi_features.csv",
    "/mnt/data/multi_hi_features.csv",   # local ChatGPT environment fallback
]

FEATURE_CSV = None
for p in CANDIDATE_FEATURE_PATHS:
    if os.path.exists(p):
        FEATURE_CSV = p
        break

# If you have a DataFrame variable named multi_hi_features in the notebook,
# you can uncomment these lines instead:
# df = multi_hi_features.copy()

if FEATURE_CSV is None:
    raise FileNotFoundError(
        "multi_hi_features.csv bulunamadı. Önce önceki Multi-HI feature extraction kodunu çalıştırın "
        "veya FEATURE_CSV değişkenini dosya yoluna göre düzenleyin."
    )

print("Loading:", FEATURE_CSV)
df = pd.read_csv(FEATURE_CSV)
print("Feature table shape:", df.shape)
print("Cells:", sorted(df["cell"].unique()))

# Ensure sorting
df = df.sort_values(["cell", "cycle_number"]).reset_index(drop=True)

In [ ]:
# ------------------------------------------------------------
# 3) METRICS AND UTILS
# ------------------------------------------------------------
def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def safe_r2(y_true, y_pred):
    try:
        return r2_score(y_true, y_pred)
    except Exception:
        return np.nan

def regression_metrics(y_true, y_pred):
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": safe_r2(y_true, y_pred),
    }

# ------------------------------------------------------------
# 4) FEATURE LEAKAGE CLEANING
# ------------------------------------------------------------
# These columns must NOT be used as model inputs because they are target-derived
# or directly define SoH.
LEAKAGE_COLUMNS_EXACT = {
    "soh", "capacity_mAh", "soh_rolling_median", "soh_delta", "soh_outlier",
    "cycle_name", "cell"
}

# We keep cycle_number because it is equivalent to EFC/cycle age and is available in practice.
# If you want to test without cycle information, set USE_CYCLE_NUMBER=False.
USE_CYCLE_NUMBER = True
USE_GLOBAL_TEMPERATURE = True

# These are acceptable because they come from charge curves / voltage windows.
# q_delta_window is a partial capacity-like feature within a voltage window, not full capacity.

def get_window_feature_cols(dataframe, window, use_cycle_number=True, use_global_temperature=True):
    """
    Return leakage-clean feature columns for a given partial-charging window.
    Only uses features from the selected window + optional cycle/temp.
    """
    suffix = "_" + window
    cols = []

    for c in dataframe.columns:
        if c in LEAKAGE_COLUMNS_EXACT:
            continue
        if c == "cycle_number" and use_cycle_number:
            cols.append(c)
        elif c in ["temp_mean", "temp_max"] and use_global_temperature:
            cols.append(c)
        elif c.endswith(suffix):
            cols.append(c)
        elif c == f"pmax_norm_{window}":
            cols.append(c)

    # Remove duplicates while preserving order
    cols = list(dict.fromkeys(cols))

    # Final leakage safety check
    bad = [c for c in cols if c in LEAKAGE_COLUMNS_EXACT or c.startswith("soh") or c == "capacity_mAh"]
    if bad:
        raise ValueError(f"Leakage columns found in feature set: {bad}")
    return cols

# Show feature columns per window
for w in WINDOWS:
    print("\n", w, "features:")
    print(get_window_feature_cols(df, w))

# Save leakage-clean feature documentation
feature_doc_rows = []
for w in WINDOWS:
    for c in get_window_feature_cols(df, w):
        feature_doc_rows.append({"window": w, "feature": c})
pd.DataFrame(feature_doc_rows).to_csv(os.path.join(RESULT_DIR, "leakage_clean_feature_sets.csv"), index=False)

In [ ]:
# ------------------------------------------------------------
# 5) TABULAR MODEL DEFINITIONS
# ------------------------------------------------------------
def make_tabular_models():
    models = {}

    # Linear models
    models["Ridge"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0, random_state=RANDOM_STATE)),
    ])
    models["ElasticNet"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNet(alpha=0.001, l1_ratio=0.2, random_state=RANDOM_STATE, max_iter=10000)),
    ])
    models["SVR-RBF"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=10.0, gamma="scale", epsilon=0.001)),
    ])
    models["KNN"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5, weights="distance")),
    ])

    # Tree and boosting models
    models["RandomForest"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", RandomForestRegressor(
            n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
        )),
    ])
    models["ExtraTrees"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", ExtraTreesRegressor(
            n_estimators=500, min_samples_leaf=1, random_state=RANDOM_STATE, n_jobs=-1
        )),
    ])
    models["HistGB"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", HistGradientBoostingRegressor(
            max_iter=400, learning_rate=0.03, l2_regularization=0.01, random_state=RANDOM_STATE
        )),
    ])

    if XGB_AVAILABLE:
        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=600, max_depth=3, learning_rate=0.02,
                subsample=0.9, colsample_bytree=0.9, reg_lambda=1.0,
                random_state=RANDOM_STATE, objective="reg:squarederror", n_jobs=-1
            )),
        ])

    if LGBM_AVAILABLE:
        models["LightGBM"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMRegressor(
                n_estimators=600, max_depth=-1, learning_rate=0.02,
                num_leaves=15, subsample=0.9, colsample_bytree=0.9,
                reg_lambda=1.0, random_state=RANDOM_STATE, verbose=-1
            )),
        ])

    if CAT_AVAILABLE:
        models["CatBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", CatBoostRegressor(
                iterations=600, depth=4, learning_rate=0.02,
                loss_function="RMSE", random_seed=RANDOM_STATE, verbose=False
            )),
        ])

    return models

# ------------------------------------------------------------
# 6) LEAVE-ONE-CELL-OUT TABULAR MULTI-HI EVALUATION
# ------------------------------------------------------------
def evaluate_tabular_multi_hi(dataframe, windows=WINDOWS, exclude_outlier_options=EXCLUDE_OUTLIER_OPTIONS):
    rows = []
    models = make_tabular_models()
    all_cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in exclude_outlier_options:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for window in windows:
            feature_cols = get_window_feature_cols(data_eval, window, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
            required = ["cell", "soh"] + feature_cols
            data_w = data_eval.dropna(subset=["cell", "soh"]).copy()

            for test_cell in all_cells:
                train_df = data_w[data_w["cell"] != test_cell].copy()
                test_df = data_w[data_w["cell"] == test_cell].copy()

                if len(test_df) < 5 or len(train_df) < 20:
                    continue

                X_train = train_df[feature_cols]
                y_train = train_df["soh"].values.astype(float)
                X_test = test_df[feature_cols]
                y_test = test_df["soh"].values.astype(float)

                for model_name, model in models.items():
                    try:
                        m = clone(model)
                        m.fit(X_train, y_train)
                        pred = m.predict(X_test)
                        pred = np.clip(pred, 0.5, 1.05)
                        met = regression_metrics(y_test, pred)
                        rows.append({
                            "experiment": "MultiHI-Tabular",
                            "window": window,
                            "test_cell": test_cell,
                            "model": model_name,
                            "exclude_outliers": exclude_outliers,
                            "n_train": len(train_df),
                            "n_test": len(test_df),
                            "n_features": len(feature_cols),
                            **met,
                        })
                    except Exception as e:
                        print(f"FAILED tabular {model_name}, {window}, {test_cell}: {e}")
    return pd.DataFrame(rows)

multi_hi_tabular_results = evaluate_tabular_multi_hi(df)
multi_hi_tabular_results.to_csv(os.path.join(RESULT_DIR, "multi_hi_tabular_leave_one_cell_results.csv"), index=False)
print("\nMulti-HI tabular results:")
display(multi_hi_tabular_results.head())

# Summary tables
multi_hi_tabular_summary = (
    multi_hi_tabular_results
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
multi_hi_tabular_summary.to_csv(os.path.join(RESULT_DIR, "multi_hi_tabular_summary_mean_std.csv"), index=False)
display(multi_hi_tabular_summary)

In [ ]:
# ------------------------------------------------------------
# 7) SINGLE-HI POLYNOMIAL EKF BASELINE
# ------------------------------------------------------------
def poly_derivative_value(coef, x):
    return np.polyval(np.polyder(coef), x)

def fit_single_hi_poly_models(train_df, window):
    """
    Prediction model: SoH = f(cycle_number), degree 2
    Measurement model: pmax_norm_window = h(SoH), degree 3
    """
    z_col = f"pmax_norm_{window}"
    tr = train_df.dropna(subset=["cycle_number", "soh", z_col]).copy()
    u = tr["cycle_number"].values.astype(float)
    x = tr["soh"].values.astype(float)
    z = tr[z_col].values.astype(float)

    pred_coef = np.polyfit(u, x, deg=2)
    meas_coef = np.polyfit(x, z, deg=3)

    pred_res = x - np.polyval(pred_coef, u)
    meas_res = z - np.polyval(meas_coef, x)
    q = max(np.var(pred_res), 1e-7)
    r = max(np.var(meas_res), 1e-7)
    return pred_coef, meas_coef, q, r

def single_hi_poly_ekf_predict(test_df, window, pred_coef, meas_coef, q, r, P0=0.01):
    z_col = f"pmax_norm_{window}"
    te = test_df.sort_values("cycle_number").dropna(subset=["cycle_number", "soh", z_col]).copy()
    preds = []
    P = P0
    x = 1.0  # BOL normalized initial health; common assumption for fresh cell

    for _, row in te.iterrows():
        u = float(row["cycle_number"])
        z = float(row[z_col])

        # Prediction from cycle degradation trend
        x_pred = float(np.polyval(pred_coef, u))
        P_pred = P + q

        # Measurement update using normalized Pmax
        z_pred = float(np.polyval(meas_coef, x_pred))
        H = float(poly_derivative_value(meas_coef, x_pred))
        S = H * P_pred * H + r
        if S <= 1e-12 or abs(H) < 1e-12:
            x = x_pred
            P = P_pred
        else:
            K = P_pred * H / S
            x = x_pred + K * (z - z_pred)
            P = (1.0 - K * H) * P_pred
        x = float(np.clip(x, 0.5, 1.05))
        preds.append(x)
    return te, np.array(preds)

def evaluate_single_hi_poly_ekf(dataframe, windows=WINDOWS, exclude_outlier_options=EXCLUDE_OUTLIER_OPTIONS):
    rows = []
    all_cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in exclude_outlier_options:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for window in windows:
            z_col = f"pmax_norm_{window}"
            if z_col not in data_eval.columns:
                continue
            for test_cell in all_cells:
                train_df = data_eval[data_eval["cell"] != test_cell].copy()
                test_df = data_eval[data_eval["cell"] == test_cell].copy()
                try:
                    pred_coef, meas_coef, q, r = fit_single_hi_poly_models(train_df, window)
                    te, pred = single_hi_poly_ekf_predict(test_df, window, pred_coef, meas_coef, q, r)
                    y_true = te["soh"].values.astype(float)
                    met = regression_metrics(y_true, pred)
                    rows.append({
                        "experiment": "SingleHI-Polynomial-EKF",
                        "window": window,
                        "test_cell": test_cell,
                        "model": "SingleHI-Polynomial-EKF",
                        "exclude_outliers": exclude_outliers,
                        "n_train": len(train_df),
                        "n_test": len(te),
                        "n_features": 1,
                        **met,
                    })
                except Exception as e:
                    print(f"FAILED SingleHI EKF {window}, {test_cell}: {e}")
    return pd.DataFrame(rows)

single_hi_ekf_results_v2 = evaluate_single_hi_poly_ekf(df)
single_hi_ekf_results_v2.to_csv(os.path.join(RESULT_DIR, "single_hi_polynomial_ekf_leave_one_cell_results.csv"), index=False)
display(single_hi_ekf_results_v2.head())

single_hi_ekf_summary_v2 = (
    single_hi_ekf_results_v2
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
single_hi_ekf_summary_v2.to_csv(os.path.join(RESULT_DIR, "single_hi_polynomial_ekf_summary_mean_std.csv"), index=False)
display(single_hi_ekf_summary_v2)

In [ ]:
# ------------------------------------------------------------
# 8) APPROXIMATE PAPER-STYLE GPR-EKF BASELINE
# ------------------------------------------------------------
def make_gpr_pipeline():
    kernel = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(length_scale=1.0, length_scale_bounds=(1e-3, 1e4)) + WhiteKernel(noise_level=1e-5, noise_level_bounds=(1e-8, 1e1))
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", GaussianProcessRegressor(
            kernel=kernel, alpha=1e-7, normalize_y=True,
            n_restarts_optimizer=1, random_state=RANDOM_STATE
        )),
    ])

def gpr_predict_mean_std(pipe, X):
    X = np.asarray(X, dtype=float)
    Xt = pipe.named_steps["imputer"].transform(X)
    Xt = pipe.named_steps["scaler"].transform(Xt)
    return pipe.named_steps["model"].predict(Xt, return_std=True)

def build_transition_training_data(train_df):
    Xs, ys = [], []
    for cell, g in train_df.sort_values(["cell", "cycle_number"]).groupby("cell"):
        g = g.dropna(subset=["soh", "cycle_number"]).copy()
        arr_soh = g["soh"].values.astype(float)
        arr_u = g["cycle_number"].values.astype(float)
        for i in range(len(g) - 1):
            x_k = arr_soh[i]
            u_k = arr_u[i]
            du = arr_u[i+1] - arr_u[i]
            y_next = arr_soh[i+1]
            Xs.append([x_k, u_k, du])
            ys.append(y_next)
    return np.asarray(Xs, dtype=float), np.asarray(ys, dtype=float)

def numerical_derivative_measurement_gpr(meas_gpr, x, delta=1e-4):
    m1, _ = gpr_predict_mean_std(meas_gpr, [[x + delta]])
    m2, _ = gpr_predict_mean_std(meas_gpr, [[x - delta]])
    return float((m1[0] - m2[0]) / (2 * delta))

def fit_approx_paper_gpr_ekf(train_df, window):
    z_col = f"pmax_norm_{window}"
    tr = train_df.dropna(subset=["soh", "cycle_number", z_col]).copy()

    # Prediction GPR: [SoH_k, u_k, delta_u] -> SoH_{k+1}
    X_pred, y_pred = build_transition_training_data(tr)
    pred_gpr = make_gpr_pipeline()
    pred_gpr.fit(X_pred, y_pred)

    # Measurement GPR: SoH_k -> normalized Pmax_k
    X_meas = tr[["soh"]].values.astype(float)
    y_meas = tr[z_col].values.astype(float)
    meas_gpr = make_gpr_pipeline()
    meas_gpr.fit(X_meas, y_meas)
    return pred_gpr, meas_gpr

def approx_paper_gpr_ekf_predict(test_df, window, pred_gpr, meas_gpr, P0=0.01):
    z_col = f"pmax_norm_{window}"
    te = test_df.sort_values("cycle_number").dropna(subset=["cycle_number", "soh", z_col]).copy()
    u_values = te["cycle_number"].values.astype(float)
    z_values = te[z_col].values.astype(float)

    preds = []
    P = P0
    x = 1.0  # Beginning-of-life normalized SoH assumption
    u_prev = float(u_values[0])

    for i, (u, z) in enumerate(zip(u_values, z_values)):
        if i == 0:
            du = 0.0
        else:
            du = float(u - u_prev)

        # Prediction step: [updated previous x, previous u, delta_u] -> predicted current x
        mean_pred, std_pred = gpr_predict_mean_std(pred_gpr, [[x, u_prev, du]])
        x_pred = float(mean_pred[0])
        q = max(float(std_pred[0]) ** 2, 1e-7)
        P_pred = P + q

        # Measurement update: h(x) -> normalized Pmax
        mean_z, std_z = gpr_predict_mean_std(meas_gpr, [[x_pred]])
        z_pred = float(mean_z[0])
        r = max(float(std_z[0]) ** 2, 1e-7)
        H = numerical_derivative_measurement_gpr(meas_gpr, x_pred)

        S = H * P_pred * H + r
        if S <= 1e-12 or abs(H) < 1e-12:
            x = x_pred
            P = P_pred
        else:
            K = P_pred * H / S
            x = x_pred + K * (z - z_pred)
            P = (1.0 - K * H) * P_pred

        x = float(np.clip(x, 0.5, 1.05))
        preds.append(x)
        u_prev = float(u)

    return te, np.array(preds)

def evaluate_approx_paper_gpr_ekf(dataframe, windows=WINDOWS, exclude_outlier_options=EXCLUDE_OUTLIER_OPTIONS):
    rows = []
    all_cells = sorted(dataframe["cell"].unique())

    for exclude_outliers in exclude_outlier_options:
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        for window in windows:
            for test_cell in all_cells:
                train_df = data_eval[data_eval["cell"] != test_cell].copy()
                test_df = data_eval[data_eval["cell"] == test_cell].copy()
                try:
                    pred_gpr, meas_gpr = fit_approx_paper_gpr_ekf(train_df, window)
                    te, pred = approx_paper_gpr_ekf_predict(test_df, window, pred_gpr, meas_gpr)
                    y_true = te["soh"].values.astype(float)
                    met = regression_metrics(y_true, pred)
                    rows.append({
                        "experiment": "ApproxPaper-GPR-EKF",
                        "window": window,
                        "test_cell": test_cell,
                        "model": "ApproxPaper-GPR-EKF",
                        "exclude_outliers": exclude_outliers,
                        "n_train": len(train_df),
                        "n_test": len(te),
                        "n_features": 1,
                        **met,
                    })
                except Exception as e:
                    print(f"FAILED ApproxPaper-GPR-EKF {window}, {test_cell}: {e}")
    return pd.DataFrame(rows)

approx_paper_gpr_ekf_results = evaluate_approx_paper_gpr_ekf(df)
approx_paper_gpr_ekf_results.to_csv(os.path.join(RESULT_DIR, "approx_paper_gpr_ekf_leave_one_cell_results.csv"), index=False)
display(approx_paper_gpr_ekf_results.head())

approx_paper_gpr_ekf_summary = (
    approx_paper_gpr_ekf_results
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
approx_paper_gpr_ekf_summary.to_csv(os.path.join(RESULT_DIR, "approx_paper_gpr_ekf_summary_mean_std.csv"), index=False)
display(approx_paper_gpr_ekf_summary)

In [ ]:
# ------------------------------------------------------------
# 9) DEEP SEQUENCE MODELS: LSTM, GRU, TCN, TRANSFORMER
# ------------------------------------------------------------
if RUN_DEEP_MODELS:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader

    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Deep learning device:", DEVICE)

    def set_torch_seed(seed=RANDOM_STATE):
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        np.random.seed(seed)
        random.seed(seed)

    class SeqDataset(Dataset):
        def __init__(self, X, y):
            self.X = torch.tensor(X, dtype=torch.float32)
            self.y = torch.tensor(y, dtype=torch.float32).view(-1, 1)
        def __len__(self):
            return len(self.y)
        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    def build_sequences(dataframe, feature_cols, seq_len=5):
        X_list, y_list, meta = [], [], []
        for cell, g in dataframe.sort_values(["cell", "cycle_number"]).groupby("cell"):
            g = g.dropna(subset=["soh"]).copy()
            X = g[feature_cols].values.astype(float)
            y = g["soh"].values.astype(float)
            cycles = g["cycle_number"].values
            for i in range(seq_len - 1, len(g)):
                X_list.append(X[i-seq_len+1:i+1])
                y_list.append(y[i])
                meta.append({"cell": cell, "cycle_number": cycles[i]})
        return np.asarray(X_list, dtype=float), np.asarray(y_list, dtype=float), pd.DataFrame(meta)

    class LSTMRegressor(nn.Module):
        def __init__(self, n_features, hidden=32, layers=1, dropout=0.1):
            super().__init__()
            self.rnn = nn.LSTM(n_features, hidden, num_layers=layers, batch_first=True,
                               dropout=dropout if layers > 1 else 0.0)
            self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        def forward(self, x):
            out, _ = self.rnn(x)
            return self.head(out[:, -1, :])

    class GRURegressor(nn.Module):
        def __init__(self, n_features, hidden=32, layers=1, dropout=0.1):
            super().__init__()
            self.rnn = nn.GRU(n_features, hidden, num_layers=layers, batch_first=True,
                              dropout=dropout if layers > 1 else 0.0)
            self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Dropout(dropout), nn.Linear(32, 1))
        def forward(self, x):
            out, _ = self.rnn(x)
            return self.head(out[:, -1, :])

    class TCNRegressor(nn.Module):
        def __init__(self, n_features, hidden=32, dropout=0.1):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(n_features, hidden, kernel_size=3, padding=1), nn.ReLU(), nn.Dropout(dropout),
                nn.Conv1d(hidden, hidden, kernel_size=3, padding=1), nn.ReLU(), nn.Dropout(dropout),
            )
            self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))
        def forward(self, x):
            # x: batch, seq, feat -> batch, feat, seq
            z = x.permute(0, 2, 1)
            z = self.net(z)
            z = z[:, :, -1]
            return self.head(z)

    class TransformerRegressor(nn.Module):
        def __init__(self, n_features, d_model=32, nhead=4, num_layers=2, dropout=0.1):
            super().__init__()
            self.input_proj = nn.Linear(n_features, d_model)
            encoder_layer = nn.TransformerEncoderLayer(
                d_model=d_model, nhead=nhead, dim_feedforward=64,
                dropout=dropout, batch_first=True, activation="gelu"
            )
            self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
            self.head = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 1))
        def forward(self, x):
            z = self.input_proj(x)
            z = self.encoder(z)
            return self.head(z[:, -1, :])

    def train_torch_regressor(model, X_train, y_train, X_val, y_val, epochs=80):
        model = model.to(DEVICE)
        train_loader = DataLoader(SeqDataset(X_train, y_train), batch_size=DEEP_BATCH_SIZE, shuffle=True)
        val_X = torch.tensor(X_val, dtype=torch.float32).to(DEVICE)
        val_y = torch.tensor(y_val, dtype=torch.float32).view(-1, 1).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=DEEP_LR, weight_decay=1e-4)
        loss_fn = nn.MSELoss()
        best_state = None
        best_val = float("inf")
        wait = 0

        for ep in range(epochs):
            model.train()
            for xb, yb in train_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                opt.zero_grad()
                loss = loss_fn(model(xb), yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            model.eval()
            with torch.no_grad():
                val_loss = float(loss_fn(model(val_X), val_y).item())
            if val_loss < best_val - 1e-8:
                best_val = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= DEEP_PATIENCE:
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        return model

    def predict_torch(model, X):
        model.eval()
        with torch.no_grad():
            X_t = torch.tensor(X, dtype=torch.float32).to(DEVICE)
            pred = model(X_t).detach().cpu().numpy().reshape(-1)
        return pred

    def evaluate_deep_models(dataframe, windows=WINDOWS, exclude_outliers=True):
        rows = []
        all_cells = sorted(dataframe["cell"].unique())
        data_eval = dataframe.copy()
        if exclude_outliers and "soh_outlier" in data_eval.columns:
            data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

        model_builders = {
            "LSTM": lambda nf: LSTMRegressor(nf),
            "GRU": lambda nf: GRURegressor(nf),
            "TCN": lambda nf: TCNRegressor(nf),
            "TransformerEncoder": lambda nf: TransformerRegressor(nf),
        }

        for window in windows:
            feature_cols = get_window_feature_cols(data_eval, window, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
            for test_cell in all_cells:
                train_df = data_eval[data_eval["cell"] != test_cell].copy()
                test_df = data_eval[data_eval["cell"] == test_cell].copy()

                # Fit imputer/scaler only on train rows to avoid leakage
                imputer = SimpleImputer(strategy="median")
                scaler = StandardScaler()
                imputer.fit(train_df[feature_cols])
                scaler.fit(imputer.transform(train_df[feature_cols]))

                train_df_proc = train_df.copy()
                test_df_proc = test_df.copy()
                train_df_proc[feature_cols] = scaler.transform(imputer.transform(train_df[feature_cols]))
                test_df_proc[feature_cols] = scaler.transform(imputer.transform(test_df[feature_cols]))

                # Split train cells into train/val by taking one validation cell from training set
                train_cells = sorted(train_df_proc["cell"].unique())
                val_cell = train_cells[-1]
                proper_train = train_df_proc[train_df_proc["cell"] != val_cell].copy()
                val_df = train_df_proc[train_df_proc["cell"] == val_cell].copy()

                X_tr, y_tr, _ = build_sequences(proper_train, feature_cols, seq_len=DEEP_SEQ_LEN)
                X_val, y_val, _ = build_sequences(val_df, feature_cols, seq_len=DEEP_SEQ_LEN)
                X_te, y_te, _ = build_sequences(test_df_proc, feature_cols, seq_len=DEEP_SEQ_LEN)

                if len(X_tr) < 20 or len(X_val) < 5 or len(X_te) < 5:
                    continue

                for model_name, builder in model_builders.items():
                    try:
                        set_torch_seed(RANDOM_STATE)
                        model = builder(len(feature_cols))
                        model = train_torch_regressor(model, X_tr, y_tr, X_val, y_val, epochs=DEEP_EPOCHS)
                        pred = predict_torch(model, X_te)
                        pred = np.clip(pred, 0.5, 1.05)
                        met = regression_metrics(y_te, pred)
                        rows.append({
                            "experiment": "MultiHI-DeepSequence",
                            "window": window,
                            "test_cell": test_cell,
                            "model": model_name,
                            "exclude_outliers": exclude_outliers,
                            "n_train": len(X_tr),
                            "n_test": len(X_te),
                            "n_features": len(feature_cols),
                            "seq_len": DEEP_SEQ_LEN,
                            **met,
                        })
                    except Exception as e:
                        print(f"FAILED deep {model_name}, {window}, {test_cell}: {e}")
        return pd.DataFrame(rows)

    deep_results = evaluate_deep_models(df, windows=WINDOWS, exclude_outliers=True)
    deep_results.to_csv(os.path.join(RESULT_DIR, "multi_hi_deep_sequence_leave_one_cell_results.csv"), index=False)
    display(deep_results.head())

    deep_summary = (
        deep_results.groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
        .agg(["mean", "std"]).reset_index()
    )
    deep_summary.to_csv(os.path.join(RESULT_DIR, "multi_hi_deep_sequence_summary_mean_std.csv"), index=False)
    display(deep_summary)
else:
    deep_results = pd.DataFrame()

In [ ]:
# ------------------------------------------------------------
# 10) COMBINE ALL RESULTS AND CREATE COMPARATIVE TABLES
# ------------------------------------------------------------
all_results_parts = [
    single_hi_ekf_results_v2,
    approx_paper_gpr_ekf_results,
    multi_hi_tabular_results,
]
if RUN_DEEP_MODELS and not deep_results.empty:
    all_results_parts.append(deep_results)

all_model_results = pd.concat(all_results_parts, ignore_index=True)
all_model_results.to_csv(os.path.join(RESULT_DIR, "all_leave_one_cell_model_results.csv"), index=False)

all_summary = (
    all_model_results
    .groupby(["exclude_outliers", "window", "experiment", "model"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
all_summary.to_csv(os.path.join(RESULT_DIR, "all_model_summary_mean_std.csv"), index=False)
print("\nALL MODEL SUMMARY:")
display(all_summary)

# Best model per window
best_per_window = (
    all_model_results.groupby(["exclude_outliers", "window", "experiment", "model"])[["RMSE", "MAE", "R2"]]
    .mean()
    .reset_index()
    .sort_values(["exclude_outliers", "window", "RMSE"])
)
best_per_window.to_csv(os.path.join(RESULT_DIR, "best_models_ranked_by_window.csv"), index=False)
display(best_per_window.groupby(["exclude_outliers", "window"]).head(10))

# Single-HI vs best Multi-HI comparison
single_avg = (
    single_hi_ekf_results_v2
    .groupby(["exclude_outliers", "window"])[["RMSE", "MAE", "R2"]]
    .mean().reset_index()
    .rename(columns={"RMSE": "single_hi_rmse", "MAE": "single_hi_mae", "R2": "single_hi_r2"})
)

multi_candidates = all_model_results[all_model_results["experiment"].isin(["MultiHI-Tabular", "MultiHI-DeepSequence"])].copy()
multi_avg = (
    multi_candidates
    .groupby(["exclude_outliers", "window", "model"])[["RMSE", "MAE", "R2"]]
    .mean().reset_index()
)
best_multi = multi_avg.sort_values(["exclude_outliers", "window", "RMSE"]).groupby(["exclude_outliers", "window"]).head(1)
best_multi = best_multi.rename(columns={"model": "best_multi_model", "RMSE": "best_multi_rmse", "MAE": "best_multi_mae", "R2": "best_multi_r2"})

single_vs_multi = pd.merge(single_avg, best_multi, on=["exclude_outliers", "window"], how="left")
single_vs_multi["rmse_improvement_percent"] = 100.0 * (single_vs_multi["single_hi_rmse"] - single_vs_multi["best_multi_rmse"]) / single_vs_multi["single_hi_rmse"]
single_vs_multi.to_csv(os.path.join(RESULT_DIR, "single_hi_vs_best_multi_hi_summary.csv"), index=False)
print("\nSingle-HI vs Best Multi-HI:")
display(single_vs_multi)

In [ ]:
# ------------------------------------------------------------
# 11) MISSING-HI ROBUSTNESS WITH REPEATED RANDOM SEEDS
# ------------------------------------------------------------
def inject_missing_in_test_only(X_test, hi_cols, missing_prob, seed):
    rng = np.random.default_rng(seed)
    X_missing = X_test.copy()
    if missing_prob <= 0:
        return X_missing
    mask = rng.random(X_missing[hi_cols].shape) < missing_prob
    X_missing.loc[:, hi_cols] = X_missing[hi_cols].mask(mask)
    return X_missing

def evaluate_missing_hi_repeated(dataframe, windows=WINDOWS, missing_probs=MISSING_PROBS, seeds=MISSING_RANDOM_SEEDS):
    rows = []
    base_models = make_tabular_models()
    # Use selected robust models for speed and clarity
    selected_names = [name for name in ["Ridge", "ExtraTrees", "HistGB", "XGBoost", "LightGBM", "CatBoost"] if name in base_models]
    all_cells = sorted(dataframe["cell"].unique())

    data_eval = dataframe.copy()
    if "soh_outlier" in data_eval.columns:
        data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

    for window in windows:
        feature_cols = get_window_feature_cols(data_eval, window, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
        # Missing only HI columns. Keep cycle number and global temp intact.
        hi_cols = [c for c in feature_cols if c not in ["cycle_number", "temp_mean", "temp_max"]]

        for model_name in selected_names:
            for missing_prob in missing_probs:
                for seed in seeds:
                    y_all, pred_all = [], []
                    for test_cell in all_cells:
                        train_df = data_eval[data_eval["cell"] != test_cell].copy()
                        test_df = data_eval[data_eval["cell"] == test_cell].copy()
                        if len(test_df) < 5:
                            continue
                        X_train = train_df[feature_cols]
                        y_train = train_df["soh"].values.astype(float)
                        X_test = test_df[feature_cols].copy()
                        y_test = test_df["soh"].values.astype(float)

                        X_test_missing = inject_missing_in_test_only(X_test, hi_cols, missing_prob, seed=seed)
                        try:
                            m = clone(base_models[model_name])
                            m.fit(X_train, y_train)
                            pred = m.predict(X_test_missing)
                            pred = np.clip(pred, 0.5, 1.05)
                            y_all.extend(y_test.tolist())
                            pred_all.extend(pred.tolist())
                        except Exception as e:
                            print(f"FAILED missing test {model_name}, {window}, p={missing_prob}, seed={seed}: {e}")

                    if len(y_all) > 0:
                        met = regression_metrics(np.asarray(y_all), np.asarray(pred_all))
                        rows.append({
                            "window": window,
                            "model": model_name,
                            "missing_prob": missing_prob,
                            "seed": seed,
                            **met,
                        })
    return pd.DataFrame(rows)

missing_hi_repeated = evaluate_missing_hi_repeated(df)
missing_hi_repeated.to_csv(os.path.join(RESULT_DIR, "missing_hi_repeated_seed_results.csv"), index=False)

missing_hi_summary_mean_std = (
    missing_hi_repeated
    .groupby(["window", "model", "missing_prob"])[["RMSE", "MAE", "R2"]]
    .agg(["mean", "std"])
    .reset_index()
)
missing_hi_summary_mean_std.to_csv(os.path.join(RESULT_DIR, "missing_hi_repeated_summary_mean_std.csv"), index=False)
print("\nMissing-HI repeated summary:")
display(missing_hi_summary_mean_std)

In [ ]:
# ------------------------------------------------------------
# 12) SHAP ANALYSIS AND PLOTS
# ------------------------------------------------------------
def train_model_for_shap(dataframe, window="medium_37_40", preferred_models=("XGBoost", "ExtraTrees", "LightGBM", "CatBoost", "RandomForest")):
    data_eval = dataframe.copy()
    if "soh_outlier" in data_eval.columns:
        data_eval = data_eval[~data_eval["soh_outlier"].astype(bool)].copy()

    feature_cols = get_window_feature_cols(data_eval, window, USE_CYCLE_NUMBER, USE_GLOBAL_TEMPERATURE)
    X = data_eval[feature_cols]
    y = data_eval["soh"].values.astype(float)
    models = make_tabular_models()
    selected = None
    for name in preferred_models:
        if name in models:
            selected = name
            break
    if selected is None:
        selected = "ExtraTrees"
    model = clone(models[selected])
    model.fit(X, y)
    return selected, model, feature_cols, data_eval

try:
    import shap
    SHAP_WINDOW = "medium_37_40"
    shap_model_name, shap_pipeline, shap_feature_cols, shap_data = train_model_for_shap(df, SHAP_WINDOW)
    print("SHAP model:", shap_model_name, "window:", SHAP_WINDOW)

    # Transform features with pipeline preprocessing.
    X_raw = shap_data[shap_feature_cols]
    imputer = shap_pipeline.named_steps.get("imputer", None)
    scaler = shap_pipeline.named_steps.get("scaler", None)
    final_model = shap_pipeline.named_steps["model"]

    X_proc = X_raw.copy()
    if imputer is not None:
        X_proc = pd.DataFrame(imputer.transform(X_proc), columns=shap_feature_cols)
    if scaler is not None:
        X_proc = pd.DataFrame(scaler.transform(X_proc), columns=shap_feature_cols)

    # Sample for faster SHAP
    if len(X_proc) > 300:
        X_shap = X_proc.sample(300, random_state=RANDOM_STATE)
    else:
        X_shap = X_proc.copy()

    explainer = shap.Explainer(final_model, X_proc)
    shap_values = explainer(X_shap)

    shap_importance = pd.DataFrame({
        "feature": shap_feature_cols,
        "mean_abs_shap": np.abs(shap_values.values).mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False)
    shap_importance.to_csv(os.path.join(RESULT_DIR, "shap_importance_advanced.csv"), index=False)
    display(shap_importance.head(20))

    # Bar plot
    plt.figure(figsize=(9, 6))
    top = shap_importance.head(15).iloc[::-1]
    plt.barh(top["feature"], top["mean_abs_shap"])
    plt.xlabel("Mean |SHAP value|")
    plt.title(f"SHAP Feature Importance - {shap_model_name} - {SHAP_WINDOW}")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "shap_bar_plot.png"), dpi=300)
    plt.show()

    # Beeswarm plot
    shap.plots.beeswarm(shap_values, max_display=15, show=False)
    plt.title(f"SHAP Beeswarm - {shap_model_name} - {SHAP_WINDOW}")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "shap_beeswarm_plot.png"), dpi=300)
    plt.show()

    # Dependence/scatter for top feature
    top_feature = shap_importance.iloc[0]["feature"]
    shap.plots.scatter(shap_values[:, top_feature], show=False)
    plt.title(f"SHAP Dependence - {top_feature}")
    plt.tight_layout()
    plt.savefig(os.path.join(RESULT_DIR, "shap_dependence_top_feature.png"), dpi=300)
    plt.show()

except Exception as e:
    print("SHAP analysis failed:", e)

In [ ]:
# ------------------------------------------------------------
# 13) EXPORT EXCEL REPORT
# ------------------------------------------------------------

def make_excel_safe(df):
    """
    Excel'e yazmadan önce MultiIndex kolonları düzleştirir.
    Örneğin:
        ('RMSE', 'mean') -> RMSE_mean
        ('RMSE', 'std')  -> RMSE_std
    """
    if df is None:
        return pd.DataFrame()

    df = df.copy()

    # MultiIndex kolonları düzleştir
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [
            "_".join([str(x) for x in col if str(x) != ""]).strip("_")
            for col in df.columns.to_flat_index()
        ]
    else:
        df.columns = [str(c) for c in df.columns]

    # MultiIndex index varsa düzleştir
    if isinstance(df.index, pd.MultiIndex):
        df = df.reset_index()
    elif df.index.name is not None:
        df = df.reset_index()

    return df


excel_path = os.path.join(RESULT_DIR, "advanced_soh_experiment_report.xlsx")

# Önceki başarısız Excel dosyası oluştuysa sil
if os.path.exists(excel_path):
    os.remove(excel_path)

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    make_excel_safe(all_model_results).to_excel(
        writer, sheet_name="all_cell_results", index=False
    )

    make_excel_safe(all_summary).to_excel(
        writer, sheet_name="summary_mean_std", index=False
    )

    make_excel_safe(best_per_window).to_excel(
        writer, sheet_name="best_models_ranked", index=False
    )

    make_excel_safe(single_vs_multi).to_excel(
        writer, sheet_name="single_vs_multi", index=False
    )

    make_excel_safe(missing_hi_repeated).to_excel(
        writer, sheet_name="missing_repeated", index=False
    )

    make_excel_safe(missing_hi_summary_mean_std).to_excel(
        writer, sheet_name="missing_mean_std", index=False
    )

    make_excel_safe(pd.DataFrame(feature_doc_rows)).to_excel(
        writer, sheet_name="feature_sets", index=False
    )

    shap_path = os.path.join(RESULT_DIR, "shap_importance_advanced.csv")
    if os.path.exists(shap_path):
        shap_df = pd.read_csv(shap_path)
        make_excel_safe(shap_df).to_excel(
            writer, sheet_name="shap_importance", index=False
        )

print("\nDONE.")
print("All results saved to Google Drive folder:", RESULT_DIR)
print("Excel report:", excel_path)
print("Important outputs:")

for fn in [
    "all_leave_one_cell_model_results.csv",
    "all_model_summary_mean_std.csv",
    "single_hi_vs_best_multi_hi_summary.csv",
    "missing_hi_repeated_summary_mean_std.csv",
    "shap_importance_advanced.csv",
    "advanced_soh_experiment_report.xlsx",
]:
    p = os.path.join(RESULT_DIR, fn)
    print(" -", p, "exists=", os.path.exists(p))
